<a href="https://colab.research.google.com/github/4kingsley/CNN-models-with-PyTorch/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Batch Processing of Blockchain Transactions
This notebook analyzes Ethereum‑like transaction data using pandas and PySpark in Google Colab.

In [1]:
!pip install pyspark pandas pyarrow matplotlib seaborn

In [3]:
from google.colab import files
uploaded = files.upload()

Saving lab1.csv to lab1.csv


In [4]:
import os
os.listdir("/content")

DATA_PATH = "/content/lab1.csv"


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BlockchainBatchAnalytics")
    .master("local[*]")
    .getOrCreate()
)

spark

In [6]:
import pandas as pd

pandas_df = pd.read_csv(DATA_PATH)
pandas_df.head()

,block_number,block_timestamp,tx_hash,from_address,to_address,value_eth,gas_used,gas_price_gwei,status,method
0,18000001,2026-05-01 10:00:00,0xaaa001,0xwallet01,0xwallet02,0.5,21000,22,1,transfer
1,18000001,2026-05-01 10:00:12,0xaaa002,0xwallet03,0xcontract01,0.0,120000,35,1,swap
2,18000002,2026-05-01 10:00:24,0xaaa003,0xwallet04,0xcontract02,0.0,85000,90,0,approve
3,18000003,2026-05-01 10:00:36,0xaaa004,0xwallet01,0xwallet05,12.0,21000,25,1,transfer
4,18000004,2026-05-01 10:00:48,0xaaa005,0xwallet06,0xcontract01,0.0,150000,110,1,swap


In [7]:
print("Rows:", len(pandas_df))
print("Total ETH value:", pandas_df["value_eth"].sum())
print("Unique senders:", pandas_df["from_address"].nunique())

Rows: 40
Total ETH value: 552.02
Unique senders: 37


In [8]:
tx_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

tx_df.printSchema()
tx_df.show(5, truncate=False)

root
 |-- block_number: integer (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)
 |-- tx_hash: string (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value_eth: double (nullable = true)
 |-- gas_used: integer (nullable = true)
 |-- gas_price_gwei: integer (nullable = true)
 |-- status: integer (nullable = true)
 |-- method: string (nullable = true)

+------------+-------------------+--------+------------+------------+---------+--------+--------------+------+--------+
|block_number|block_timestamp    |tx_hash |from_address|to_address  |value_eth|gas_used|gas_price_gwei|status|method  |
+------------+-------------------+--------+------------+------------+---------+--------+--------------+------+--------+
|18000001    |2026-05-01 10:00:00|0xaaa001|0xwallet01  |0xwallet02  |0.5      |21000   |22            |1     |transfer|
|18000001    |2026-05-01 10:00:12|0xaaa002|0xwallet03  |0xcontract01|0.0      |120000  |

In [9]:
from pyspark.sql import functions as F

# Count missing values per column
tx_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in tx_df.columns
]).show()

# Remove duplicate transactions by tx_hash
tx_clean = tx_df.dropDuplicates(["tx_hash"])

# Check status distribution (success vs failure)
tx_clean.groupBy("status").count().show()

+------------+---------------+-------+------------+----------+---------+--------+--------------+------+------+
|block_number|block_timestamp|tx_hash|from_address|to_address|value_eth|gas_used|gas_price_gwei|status|method|
+------------+---------------+-------+------------+----------+---------+--------+--------------+------+------+
|           0|              0|      0|           0|         0|        0|       0|             0|     0|     0|
+------------+---------------+-------+------------+----------+---------+--------+--------------+------+------+

+------+-----+
|status|count|
+------+-----+
|     1|   34|
|     0|    6|
+------+-----+



In [10]:
tx_enriched = tx_clean.withColumn(
    "gas_fee_eth",
    (F.col("gas_used") * F.col("gas_price_gwei")) / F.lit(1_000_000_000)
)

tx_enriched.select(
    "tx_hash", "gas_used", "gas_price_gwei", "gas_fee_eth"
).show(truncate=False)


+--------+--------+--------------+-----------+
|tx_hash |gas_used|gas_price_gwei|gas_fee_eth|
+--------+--------+--------------+-----------+
|0xaaa009|160000  |130           |0.0208     |
|0xaaa037|110000  |60            |0.0066     |
|0xaaa004|21000   |25            |5.25E-4    |
|0xaaa030|90000   |85            |0.00765    |
|0xaaa010|21000   |19            |3.99E-4    |
|0xaaa035|21000   |23            |4.83E-4    |
|0xaaa005|150000  |110           |0.0165     |
|0xaaa007|95000   |42            |0.00399    |
|0xaaa014|200000  |95            |0.019      |
|0xaaa003|85000   |90            |0.00765    |
|0xaaa016|21000   |21            |4.41E-4    |
|0xaaa023|175000  |140           |0.0245     |
|0xaaa039|80000   |95            |0.0076     |
|0xaaa038|195000  |155           |0.030225   |
|0xaaa031|170000  |145           |0.02465    |
|0xaaa006|21000   |24            |5.04E-4    |
|0xaaa018|190000  |120           |0.0228     |
|0xaaa026|125000  |55            |0.006875   |
|0xaaa032|210

In [11]:
tx_by_block = (
    tx_enriched
    .groupBy("block_number")
    .agg(
        F.count("*").alias("tx_count"),
        F.sum("value_eth").alias("total_value_eth"),
        F.avg("gas_price_gwei").alias("avg_gas_price_gwei")
    )
    .orderBy("block_number")
)

tx_by_block.show()


+------------+--------+---------------+------------------+
|block_number|tx_count|total_value_eth|avg_gas_price_gwei|
+------------+--------+---------------+------------------+
|    18000001|       2|            0.5|              28.5|
|    18000002|       1|            0.0|              90.0|
|    18000003|       1|           12.0|              25.0|
|    18000004|       1|            0.0|             110.0|
|    18000005|       1|           1.25|              24.0|
|    18000006|       1|            0.0|              42.0|
|    18000007|       1|           25.0|              75.0|
|    18000008|       1|            0.0|             130.0|
|    18000009|       1|           0.05|              19.0|
|    18000010|       1|            0.0|             160.0|
|    18000011|       1|            4.8|              30.0|
|    18000012|       1|           35.0|              28.0|
|    18000013|       1|            0.0|              95.0|
|    18000014|       1|            0.0|              88.

In [12]:
top_senders = (
    tx_enriched
    .groupBy("from_address")
    .agg(
        F.count("*").alias("sent_tx_count"),
        F.sum("value_eth").alias("total_sent_eth"),
        F.sum("gas_fee_eth").alias("total_gas_fee_eth")
    )
    .orderBy(F.desc("sent_tx_count"))
)

top_senders.show(truncate=False)


+------------+-------------+--------------+-----------------+
|from_address|sent_tx_count|total_sent_eth|total_gas_fee_eth|
+------------+-------------+--------------+-----------------+
|0xwallet01  |4            |47.5          |0.026225         |
|0xwallet31  |1            |0.0           |0.006875         |
|0xwallet43  |1            |0.0           |0.022275         |
|0xwallet07  |1            |1.25          |5.04E-4          |
|0xwallet06  |1            |0.0           |0.0165           |
|0xwallet03  |1            |0.0           |0.0042           |
|0xwallet44  |1            |0.0           |0.0066           |
|0xwallet04  |1            |0.0           |0.00765          |
|0xwallet10  |1            |25.0          |0.0135           |
|0xwallet47  |1            |0.15          |3.57E-4          |
|0xwallet16  |1            |0.0           |0.019            |
|0xwallet22  |1            |0.0           |0.045            |
|0xwallet23  |1            |0.1           |4.2E-4           |
|0xwalle

In [13]:
top_receivers = (
    tx_enriched
    .groupBy("to_address")
    .agg(
        F.count("*").alias("received_tx_count"),
        F.sum("value_eth").alias("total_received_eth")
    )
    .orderBy(F.desc("received_tx_count"))
)

top_receivers.show(truncate=False)


+------------+-----------------+------------------+
|to_address  |received_tx_count|total_received_eth|
+------------+-----------------+------------------+
|0xcontract01|7                |0.0               |
|0xcontract02|3                |0.0               |
|0xbridge01  |2                |125.0             |
|0xcontract04|2                |0.0               |
|0xbridge02  |2                |275.0             |
|0xexchange01|2                |53.5              |
|0xwallet08  |1                |1.25              |
|0xcontract10|1                |0.0               |
|0xcontract05|1                |0.0               |
|0xwallet05  |1                |12.0              |
|0xcontract08|1                |0.0               |
|0xwallet48  |1                |0.15              |
|0xwallet12  |1                |0.05              |
|0xwallet38  |1                |3.7               |
|0xcontract11|1                |0.0               |
|0xwallet15  |1                |4.8               |
|0xcontract0

In [14]:
failed_tx = tx_enriched.filter(F.col("status") == 0)

failed_by_receiver = (
    failed_tx
    .groupBy("to_address")
    .agg(F.count("*").alias("failed_count"))
    .orderBy(F.desc("failed_count"))
)

failed_by_receiver.show(truncate=False)


+------------+------------+
|to_address  |failed_count|
+------------+------------+
|0xcontract02|3           |
|0xcontract04|2           |
|0xcontract12|1           |
+------------+------------+



In [15]:
anomalies = tx_enriched.filter(
    (F.col("value_eth") > 10) |
    (F.col("gas_price_gwei") > 100)
)

anomalies.select(
    "tx_hash", "from_address", "to_address",
    "value_eth", "gas_price_gwei", "method", "status"
).show(truncate=False)


+--------+------------+------------+---------+--------------+--------------+------+
|tx_hash |from_address|to_address  |value_eth|gas_price_gwei|method        |status|
+--------+------------+------------+---------+--------------+--------------+------+
|0xaaa004|0xwallet01  |0xwallet05  |12.0     |25            |transfer      |1     |
|0xaaa005|0xwallet06  |0xcontract01|0.0      |110           |swap          |1     |
|0xaaa008|0xwallet10  |0xbridge01  |25.0     |75            |bridge_deposit|1     |
|0xaaa009|0xwallet02  |0xcontract01|0.0      |130           |swap          |1     |
|0xaaa011|0xwallet13  |0xcontract04|0.0      |160           |swap          |0     |
|0xaaa013|0xwallet01  |0xexchange01|35.0     |28            |transfer      |1     |
|0xaaa017|0xwallet20  |0xcontract01|0.0      |105           |swap          |1     |
|0xaaa018|0xwallet21  |0xbridge01  |100.0    |120           |bridge_deposit|1     |
|0xaaa019|0xwallet22  |0xcontract06|0.0      |180           |contract_call |

In [16]:
tx_enriched.write.mode("overwrite").parquet("/content/eth_transactions_enriched.parquet")

In [17]:
parquet_df = spark.read.parquet("/content/eth_transactions_enriched.parquet")
parquet_df.show(5, truncate=False)

+------------+-------------------+--------+------------+------------+---------+--------+--------------+------+--------+-----------+
|block_number|block_timestamp    |tx_hash |from_address|to_address  |value_eth|gas_used|gas_price_gwei|status|method  |gas_fee_eth|
+------------+-------------------+--------+------------+------------+---------+--------+--------------+------+--------+-----------+
|18000001    |2026-05-01 10:00:00|0xaaa001|0xwallet01  |0xwallet02  |0.5      |21000   |22            |1     |transfer|4.62E-4    |
|18000001    |2026-05-01 10:00:12|0xaaa002|0xwallet03  |0xcontract01|0.0      |120000  |35            |1     |swap    |0.0042     |
|18000002    |2026-05-01 10:00:24|0xaaa003|0xwallet04  |0xcontract02|0.0      |85000   |90            |0     |approve |0.00765    |
|18000003    |2026-05-01 10:00:36|0xaaa004|0xwallet01  |0xwallet05  |12.0     |21000   |25            |1     |transfer|5.25E-4    |
|18000004    |2026-05-01 10:00:48|0xaaa005|0xwallet06  |0xcontract01|0.0    

In [18]:
!zip -r eth_transactions_enriched_parquet.zip /content/eth_transactions_enriched.parquet


  adding: content/eth_transactions_enriched.parquet/ (stored 0%)
  adding: content/eth_transactions_enriched.parquet/._SUCCESS.crc (stored 0%)
  adding: content/eth_transactions_enriched.parquet/part-00000-980da563-0bfb-48d1-8729-5d0a722cc88b-c000.snappy.parquet (deflated 46%)
  adding: content/eth_transactions_enriched.parquet/.part-00000-980da563-0bfb-48d1-8729-5d0a722cc88b-c000.snappy.parquet.crc (stored 0%)
  adding: content/eth_transactions_enriched.parquet/_SUCCESS (stored 0%)


In [19]:
from google.colab import files
files.download("eth_transactions_enriched_parquet.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from pyspark.sql import functions as F

total_rows = tx_enriched.count()
unique_tx = tx_enriched.select("tx_hash").distinct().count()
unique_senders = tx_enriched.select("from_address").distinct().count()
unique_receivers = tx_enriched.select("to_address").distinct().count()

min_block = tx_enriched.agg(F.min("block_number")).first()[0]
max_block = tx_enriched.agg(F.max("block_number")).first()[0]

min_time = tx_enriched.agg(F.min("block_timestamp")).first()[0]
max_time = tx_enriched.agg(F.max("block_timestamp")).first()[0]

print("Rows:", total_rows)
print("Unique transactions:", unique_tx)
print("Unique senders:", unique_senders)
print("Unique receivers:", unique_receivers)
print("Block range:", min_block, "→", max_block)
print("Timestamp range:", min_time, "→", max_time)


Rows: 40
Unique transactions: 40
Unique senders: 37
Unique receivers: 28
Block range: 18000001 → 18000039
Timestamp range: 2026-05-01 10:00:00 → 2026-05-01 10:07:48


In [21]:
method_analysis = (
    tx_enriched
    .groupBy("method")
    .agg(
        F.count("*").alias("tx_count"),
        F.sum("value_eth").alias("total_value_eth"),
        F.avg("gas_price_gwei").alias("avg_gas_price_gwei")
    )
    .orderBy(F.desc("tx_count"))
)

method_analysis.show(truncate=False)


+--------------+--------+---------------+------------------+
|method        |tx_count|total_value_eth|avg_gas_price_gwei|
+--------------+--------+---------------+------------------+
|transfer      |15      |152.02         |24.266666666666666|
|swap          |9       |0.0            |123.33333333333333|
|approve       |6       |0.0            |74.16666666666667 |
|bridge_deposit|4       |400.0          |116.25            |
|contract_call |2       |0.0            |190.0             |
|nft_purchase  |2       |0.0            |132.5             |
|claim         |1       |0.0            |60.0              |
|stake         |1       |0.0            |55.0              |
+--------------+--------+---------------+------------------+



In [22]:
status_analysis = (
    tx_enriched
    .groupBy("status")
    .agg(
        F.count("*").alias("tx_count"),
        F.avg("gas_used").alias("avg_gas_used"),
        F.avg("gas_price_gwei").alias("avg_gas_price_gwei")
    )
)

status_analysis.show()


+------+--------+-----------------+------------------+
|status|tx_count|     avg_gas_used|avg_gas_price_gwei|
+------+--------+-----------------+------------------+
|     1|      34|103823.5294117647| 72.82352941176471|
|     0|       6|         104500.0|111.33333333333333|
+------+--------+-----------------+------------------+



In [23]:
wallet_activity = (
    tx_enriched
    .groupBy("from_address")
    .agg(
        F.count("*").alias("sent_tx_count"),
        F.sum("value_eth").alias("total_sent_eth"),
        F.sum("gas_fee_eth").alias("total_gas_fee_eth")
    )
)

wallet_activity.orderBy(F.desc("sent_tx_count")).show(5, truncate=False)
wallet_activity.orderBy(F.desc("total_sent_eth")).show(5, truncate=False)


+------------+-------------+--------------+-----------------+
|from_address|sent_tx_count|total_sent_eth|total_gas_fee_eth|
+------------+-------------+--------------+-----------------+
|0xwallet01  |4            |47.5          |0.026225         |
|0xwallet31  |1            |0.0           |0.006875         |
|0xwallet43  |1            |0.0           |0.022275         |
|0xwallet07  |1            |1.25          |5.04E-4          |
|0xwallet06  |1            |0.0           |0.0165           |
+------------+-------------+--------------+-----------------+
only showing top 5 rows
+------------+-------------+--------------+-----------------+
|from_address|sent_tx_count|total_sent_eth|total_gas_fee_eth|
+------------+-------------+--------------+-----------------+
|0xwallet45  |1            |220.0         |0.030225         |
|0xwallet21  |1            |100.0         |0.0228           |
|0xwallet39  |1            |72.0          |6.51E-4          |
|0xwallet35  |1            |55.0          |0.0

In [24]:
gas_stats = tx_enriched.agg(
    F.min("gas_price_gwei").alias("min_gas_price"),
    F.max("gas_price_gwei").alias("max_gas_price"),
    F.avg("gas_price_gwei").alias("avg_gas_price"),
    F.min("gas_fee_eth").alias("min_gas_fee_eth"),
    F.max("gas_fee_eth").alias("max_gas_fee_eth"),
    F.avg("gas_fee_eth").alias("avg_gas_fee_eth")
)

gas_stats.show()


+-------------+-------------+-------------+---------------+---------------+--------------------+
|min_gas_price|max_gas_price|avg_gas_price|min_gas_fee_eth|max_gas_fee_eth|     avg_gas_fee_eth|
+-------------+-------------+-------------+---------------+---------------+--------------------+
|           17|          200|         78.6|        3.57E-4|           0.06|0.012034124999999998|
+-------------+-------------+-------------+---------------+---------------+--------------------+



In [25]:
avg_gas = tx_enriched.agg(F.avg("gas_price_gwei")).first()[0]

high_gas_tx = tx_enriched.filter(F.col("gas_price_gwei") > avg_gas)

high_gas_tx.select(
    "tx_hash", "from_address", "to_address",
    "method", "gas_price_gwei", "gas_fee_eth"
).show(truncate=False)


+--------+------------+------------+--------------+--------------+-----------+
|tx_hash |from_address|to_address  |method        |gas_price_gwei|gas_fee_eth|
+--------+------------+------------+--------------+--------------+-----------+
|0xaaa003|0xwallet04  |0xcontract02|approve       |90            |0.00765    |
|0xaaa005|0xwallet06  |0xcontract01|swap          |110           |0.0165     |
|0xaaa009|0xwallet02  |0xcontract01|swap          |130           |0.0208     |
|0xaaa011|0xwallet13  |0xcontract04|swap          |160           |0.0224     |
|0xaaa014|0xwallet16  |0xcontract05|nft_purchase  |95            |0.019      |
|0xaaa015|0xwallet17  |0xcontract02|approve       |88            |0.007656   |
|0xaaa017|0xwallet20  |0xcontract01|swap          |105           |0.016275   |
|0xaaa018|0xwallet21  |0xbridge01  |bridge_deposit|120           |0.0228     |
|0xaaa019|0xwallet22  |0xcontract06|contract_call |180           |0.045      |
|0xaaa023|0xwallet27  |0xcontract01|swap          |1

In [26]:
anomalies_ext = tx_enriched.filter(
    (F.col("value_eth") > 20) |
    (F.col("gas_price_gwei") > 120) |
    (F.col("gas_used") > 180000) |
    ((F.col("status") == 0) & (F.col("gas_price_gwei") > 80))
)

anomalies_ext = anomalies_ext.withColumn(
    "anomaly_reason",
    F.when(F.col("value_eth") > 20, "high value")
     .when(F.col("gas_price_gwei") > 120, "high gas price")
     .when(F.col("gas_used") > 180000, "high gas used")
     .when((F.col("status") == 0) & (F.col("gas_price_gwei") > 80), "failed + high gas")
)

anomalies_ext.select(
    "tx_hash", "from_address", "to_address", "method",
    "value_eth", "gas_used", "gas_price_gwei", "status", "anomaly_reason"
).show(truncate=False)


+--------+------------+------------+--------------+---------+--------+--------------+------+-----------------+
|tx_hash |from_address|to_address  |method        |value_eth|gas_used|gas_price_gwei|status|anomaly_reason   |
+--------+------------+------------+--------------+---------+--------+--------------+------+-----------------+
|0xaaa003|0xwallet04  |0xcontract02|approve       |0.0      |85000   |90            |0     |failed + high gas|
|0xaaa008|0xwallet10  |0xbridge01  |bridge_deposit|25.0     |180000  |75            |1     |high value       |
|0xaaa009|0xwallet02  |0xcontract01|swap          |0.0      |160000  |130           |1     |high gas price   |
|0xaaa011|0xwallet13  |0xcontract04|swap          |0.0      |140000  |160           |0     |high gas price   |
|0xaaa013|0xwallet01  |0xexchange01|transfer      |35.0     |21000   |28            |1     |high value       |
|0xaaa014|0xwallet16  |0xcontract05|nft_purchase  |0.0      |200000  |95            |1     |high gas used    |
|

In [27]:
anomalies_ext.groupBy("method").count().show()
method_analysis.write.mode("overwrite").parquet("/content/method_analysis.parquet")
anomalies_ext.write.mode("overwrite").parquet("/content/anomaly_detection.parquet")

# Read back
spark.read.parquet("/content/method_analysis.parquet").show(5)
spark.read.parquet("/content/anomaly_detection.parquet").show(5)


+--------------+-----+
|        method|count|
+--------------+-----+
| contract_call|    2|
|  nft_purchase|    2|
|          swap|    6|
|       approve|    4|
|bridge_deposit|    4|
|      transfer|    2|
+--------------+-----+

+--------------+--------+---------------+------------------+
|        method|tx_count|total_value_eth|avg_gas_price_gwei|
+--------------+--------+---------------+------------------+
|      transfer|      15|         152.02|24.266666666666666|
|          swap|       9|            0.0|123.33333333333333|
|       approve|       6|            0.0| 74.16666666666667|
|bridge_deposit|       4|          400.0|            116.25|
| contract_call|       2|            0.0|             190.0|
+--------------+--------+---------------+------------------+
only showing top 5 rows
+------------+-------------------+--------+------------+------------+---------+--------+--------------+------+--------------+-----------+-----------------+
|block_number|    block_timestamp| tx_ha